In [4]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr


c:\project\RAG_PDF_READER\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
MODEL = "gemini-3.1-flash-lite"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("sample.pdf")
documents = loader.load()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_27032\3503506097.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

In [8]:
embeddings = HuggingFaceEmbeddings()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1027.82it/s]


In [9]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME
)

In [10]:
print("Number of documents:", len(documents))

Number of documents: 21


In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0
)

In [12]:
retriever = vectorstore.as_retriever()

In [24]:
llm = ChatOpenAI(
    model="llama3.2",
    temperature=0,
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)
messages = [
    SystemMessage(content="You are a helpful assistant. Answer based on the provided context."),
    HumanMessage(content="What is electric charge?")
]
response = llm.invoke(messages)

print(response.content)

Electric charge is a fundamental property of matter that causes it to interact with other charged particles and objects. It is a measure of the amount of energy an object possesses due to its presence or absence of electrons.

There are two types of electric charges:

1. **Positive charge**: This type of charge is found in protons, which are positively charged subatomic particles that reside in the nucleus of an atom.
2. **Negative charge**: This type of charge is found in electrons, which are negatively charged subatomic particles that orbit the nucleus of an atom.

When two objects have opposite charges (one positive and one negative), they attract each other. When two objects have the same type of charge (both positive or both negative), they repel each other.

The magnitude of electric charge is measured in units called coulombs (C). The unit of measurement for electric charge was introduced by French physicist Charles-Augustin de Coulomb, from whom it gets its name.

Electric char

In [19]:
retriever.invoke("What is charge")

[Document(id='2bc120ff-a78a-4311-bd20-7439712ec22d', metadata={'total_pages': 21, 'page': 1, 'creator': 'Chromium', 'source': 'sample.pdf', 'producer': 'Skia/PDF m136', 'title': 'Electric Charges and Fields - Textbook', 'page_label': '2', 'moddate': '2026-09-20T14:50:17+00:00', 'creationdate': '2026-09-20T14:50:17+00:00'}, page_content='created nor destroyed; it can only be transferred from one object to another or created/annihilated in equal positive-\nnegative pairs.\nConservation Law: In any closed, isolated physical process, the algebraic sum of all electric charges remains\nperfectly invariant.\n2.4 Quantization of Charge\nElectric charge is not continuous; it exists in discrete packets. Any free macroscopic or microscopic charge q in nature\nis always an integral multiple of the fundamental unit of charge e.\nq = \\pm n e \\quad \\text{where } n \\in \\mathbb{Z}^+ \\text{ and } e = 1.602176 \\times 10^{-19}\\text{ C}\nConceptual Example 2.1: Can a body carry a net charge of 4.0 

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """

You are a knowledgeable and friendly Physics assistant.

You are helping the user understand the topic "Electric Charges and Fields".

Use the provided context from the study material to answer the user's questions.

Your answers should be:
- Clear and easy to understand
- Scientifically accurate
- Based primarily on the provided context
- Appropriate for a student learning Physics

You can explain concepts, definitions, formulas, laws, derivations, and numerical concepts related to Electric Charges and Fields.

If a formula is relevant, explain what each symbol represents and mention the SI unit when appropriate.

If the answer is not available in the provided context, say:
"I don't know based on the provided study material."

Do not make up information.

Context:

{context}

"""

In [21]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [17]:
answer_question("Concept of Electric Charge", [])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text',
  'text': 'Based on the provided study material, here is an explanation of the concept of electric charge:\n\n### **What is Electric Charge?**\nElectric charge ($q$ or $Q$) is an **intrinsic physical property** of microscopic matter. It is one of the fundamental attributes of subatomic particles, similar to mass. \n\nWhile mass is always positive and results in gravitational interactions, electric charge exists in two distinct types and is responsible for both **attractive and repulsive forces** when matter is placed in an electromagnetic field.\n\n### **Key Characteristics:**\n*   **Scalar Property:** Electric charge is a scalar quantity, meaning it has magnitude but no direction.\n*   **SI Unit:** The SI unit of electric charge is the **Coulomb (C)**.\n*   **Fundamental Nature:** It determines how matter interacts electromagnetically.\n\n### **Core Properties:**\nThe behavior of electric charge is governed by three fundamental principles:\n1.  **Additivity:** Becaus

In [25]:
import gradio as gr

demo = gr.ChatInterface(
    fn=answer_question,
    title="Electric Charges & Fields Assistant"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
